In [ ]:
import sys, os
import numpy as np
import pandas as pd
from scipy import stats

from csc import *
from exp_utils import *

current_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tol_colors as tc

In [4]:
import scienceplots
plt.style.use(['science', 'no-latex'])
plt.rcParams['text.latex.preamble'] = r'\usepackage[cm]{sfmath}'
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'

Run `collect_uncertainty.py` before this notebook.

In [43]:
potato_duplicate_questions = [14, 83, 121]
models = [
    "gemma-2-9b-it",
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Phi-3.5-mini-instruct",
]

model_rename = {
    "gemma-2-9b-it": "Gemma-2-9B",
    "gemma-3-12b-it": "Gemma-3-12B",
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B",
    "Phi-3.5-mini-instruct": "Phi-3.5-3.8B",
}

entropy_methods = {
    "plugin": "$\\widehat{\\mathbb{H}}_{Plugin}$", # uses NumSets
    "cs": "$\\widehat{\\mathbb{H}}_{CS-GT}$", # uses GT
    "cs-hybrid": "$\\widehat{\\mathbb{H}}_{Hybrid}$", # uses H
}

datasets = {
    "hotpot_qa_final": "HotpotQA",
    "squad_v2_final": "SQuAD 2.0",
    "potato_final": "POTATO",
    "bioasq_final": "BioASQ",
}
pm_symbol = u"\u00B1"
num_samples_list = [5, 10, 25, 50, 75, 100]

#### detailed entropy ratio results

In [44]:
p = "no_preprompt"

In [ ]:
square = False

if square:
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 1])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
else:
    fig = plt.figure(figsize=(25, 5))
    gs = gridspec.GridSpec(1, 5, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    ax5 = fig.add_subplot(gs[0, 4])
    
axes = [ax1, ax2, ax3, ax4, ax5]

colors = tc.get_colorset('muted')
dataset_colors = {
    "HotpotQA": colors[0],
    "SQuAD 2.0": colors[1],
    "POTATO": colors[3],
    "BioASQ": colors[4]
}

large_fontsize = 35
medium_fontsize = 30
small_fontsize = 25

for model_idx in range(len(models)):
    model = models[model_idx]
    ax = axes[model_idx]
    ax.axhline(
        y=1., 
        linestyle=':', 
        color='grey', 
        label='$\\langle\widehat{SE}/SE^*\\rangle=1$'
    )

    entropy_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/uncertainty.csv")

    for dataset in datasets:
        means_hybrid = []
        means_plugin = []
        
        for n in num_samples_list:
            df_subset = entropy_df[(entropy_df["dataset"]==dataset)&(entropy_df["n"]==n)&(entropy_df["oracle"]>0)]
            ratios_hybrid = df_subset["cs-hybrid"]/df_subset["oracle"]
            ratios_plugin = df_subset["plugin"]/df_subset["oracle"]
            ratios_oracle = df_subset["plugin"]/df_subset["oracle"]
            means_hybrid.append(ratios_hybrid.mean())
            means_plugin.append(ratios_plugin.mean())

        ax.plot(
            num_samples_list, means_hybrid, "o",
            label=datasets[dataset] + " ($\\widehat{\\mathbb{H}}_{Hybrid}$)",
            color=dataset_colors[datasets[dataset]],
            linestyle=None,
            lw=3
        )
        ax.plot(
            num_samples_list, means_plugin, "x",
            label=datasets[dataset] + " ($\\widehat{\\mathbb{H}}_{Plugin}$)", 
            color=dataset_colors[datasets[dataset]],
            linestyle=":",
            lw=3
        )
        sns.despine(top=True, right=True, left=False, bottom=False, ax=ax)
        ax.xaxis.set_minor_locator(plt.NullLocator())
        ax.yaxis.set_minor_locator(plt.NullLocator())
        ax.tick_params(axis="y", which="both", right=False)
        ax.tick_params(axis="x", which="both", top=False) 

    ax.set_title(model_rename[model], fontsize=large_fontsize)
    ax.set_xlabel("$n$", fontsize=large_fontsize)
    if model_idx == 0:
        ax.set_ylabel("$\\langle\widehat{SE}/SE^*\\rangle$", fontsize=large_fontsize)
    ax.set_ylim(0.3, 2)
    ax.set_xscale("log")
    ax.set_xticks([5, 10, 25, 50, 100])
    ax.set_xticklabels([5, 10, 25, 50, 100])
    ax.set_yticks([0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2])
    ax.tick_params(axis='both', which='both', labelsize=small_fontsize)


handles, labels = ax.get_legend_handles_labels()
handles = handles[1:]+[handles[0]]
labels = labels[1:]+[labels[0]]
# interleave for 2 column setup
if square:
    handles = [handles[i] for i in range(1, len(handles), 2)]+[handles[i] for i in range(0, len(handles), 2)]
    labels = [labels[i] for i in range(1, len(labels), 2)]+[labels[i] for i in range(0, len(labels), 2)]
fig.legend(
    handles, labels, fontsize=small_fontsize, loc='lower center', 
        bbox_to_anchor=(0.5, -0.3), ncol=5
)

plt.tight_layout()
plt.savefig('figures/entropy_ratios_plugin_draft.pdf')

#### aggregated entropy ratio results

In [ ]:
square = False

if square:
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 1])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
else:
    fig = plt.figure(figsize=(25, 5))
    gs = gridspec.GridSpec(1, 5, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    ax5 = fig.add_subplot(gs[0, 4])
    
axes = [ax1, ax2, ax3, ax4, ax5]

colors = tc.get_colorset('muted')
dataset_colors = {
    "HotpotQA": colors[0],
    "SQuAD 2.0": colors[1],
    "POTATO": colors[3],
    "BioASQ": colors[4]
}

large_fontsize = 35
medium_fontsize = 30
small_fontsize = 25

for model_idx in range(len(models)):
    model = models[model_idx]
    ax = axes[model_idx]
    ax.axhline(
        y=1., 
        linestyle=':', 
        color='grey', 
        label='$\\langle\widehat{SE}/SE^*\\rangle=1$'
    )

    entropy_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/uncertainty.csv")

    aggregate_means_hybrid = None
    aggregate_means_plugin = None
    aggregate_means_se = None

    for dataset in datasets:
        means_hybrid = []
        means_plugin = []
        means_se = []
        
        for n in num_samples_list:
            df_subset = entropy_df[(entropy_df["dataset"]==dataset)&(entropy_df["n"]==n)&(entropy_df["oracle"]>0)]
            ratios_hybrid = df_subset["cs-hybrid"]/df_subset["oracle"]
            ratios_plugin = df_subset["plugin"]/df_subset["oracle"]
            ratios_se = df_subset["se"]/df_subset["oracle"]

            means_hybrid.append(ratios_hybrid.mean())
            means_plugin.append(ratios_plugin.mean())
            means_se.append(ratios_se.mean())
        
        if aggregate_means_hybrid is None:
            aggregate_means_hybrid = np.array(means_hybrid)
        else:
            aggregate_means_hybrid += np.array(means_hybrid)

        if aggregate_means_plugin is None:
            aggregate_means_plugin = np.array(means_plugin)
        else:
            aggregate_means_plugin += np.array(means_plugin)

        if aggregate_means_se is None:
            aggregate_means_se = np.array(means_se)
        else:
            aggregate_means_se += np.array(means_se)

    aggregate_means_hybrid /= len(datasets)
    aggregate_means_plugin /= len(datasets)
    aggregate_means_se /= len(datasets)

    ax.plot(
        num_samples_list, aggregate_means_hybrid, 
        "o", markersize=12,
        label="$\\widehat{\\mathbb{H}}_{Hybrid}$ (Ours)",
        color=colors.indigo,
        linestyle=None,
        lw=4
    )

    ax.plot(
        num_samples_list, aggregate_means_plugin, 
        "x", markersize=12,
        label="$\\widehat{\\mathbb{H}}_{Plugin}$ (Canonical DSE)",
        color=colors.indigo,
        linestyle=":",
        lw=4
    )

    sns.despine(top=True, right=True, left=False, bottom=False, ax=ax)
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.tick_params(axis="y", which="both", right=False)
    ax.tick_params(axis="x", which="both", top=False) 

    ax.set_title(model_rename[model], fontsize=large_fontsize)
    ax.set_xlabel("$n$", fontsize=large_fontsize)
    if model_idx == 0:
        ax.set_ylabel("$\\langle\widehat{SE}/SE^*\\rangle$", fontsize=large_fontsize)
    ax.set_ylim(0.3, 1.2)
    ax.set_xscale("log")
    ax.set_xticks([5, 10, 25, 50, 100])
    ax.set_xticklabels([5, 10, 25, 50, 100])
    ax.set_yticks([0.4, 0.6, 0.8, 1.0, 1.2])
    ax.tick_params(axis='both', which='both', labelsize=small_fontsize)

handles, labels = ax.get_legend_handles_labels()
handles = [handles[1], handles[2], handles[0]]
labels = [labels[1], labels[2], labels[0]]
# interleave for 2 column setup
if square:
    handles = [handles[i] for i in range(1, len(handles), 2)]+[handles[i] for i in range(0, len(handles), 2)]
    labels = [labels[i] for i in range(1, len(labels), 2)]+[labels[i] for i in range(0, len(labels), 2)]
fig.legend(
    handles, labels, fontsize=medium_fontsize, loc='lower center', 
        bbox_to_anchor=(0.5, -0.35), ncol=3
)

plt.tight_layout()
plt.savefig('figures/entropy_ratios_plugin_summary_draft.pdf')

#### version of above figure (aggregated entropy ratio results) including SE

In [ ]:
square = False

if square:
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 1])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
else:
    fig = plt.figure(figsize=(25, 5))
    gs = gridspec.GridSpec(1, 5, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    ax5 = fig.add_subplot(gs[0, 4])
    
axes = [ax1, ax2, ax3, ax4, ax5]

colors = tc.get_colorset('muted')
dataset_colors = {
    "HotpotQA": colors[0],
    "SQuAD 2.0": colors[1],
    "POTATO": colors[3],
    "BioASQ": colors[4]
}

large_fontsize = 35
medium_fontsize = 30
small_fontsize = 25

for model_idx in range(len(models)):
    model = models[model_idx]
    ax = axes[model_idx]
    ax.axhline(
        y=1., 
        linestyle=':', 
        color='grey', 
        label='$\\langle\widehat{SE}/SE^*\\rangle=1$'
    )

    entropy_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/uncertainty.csv")

    aggregate_means_hybrid = None
    aggregate_means_plugin = None
    aggregate_means_se = None

    for dataset in datasets:
        means_hybrid = []
        means_plugin = []
        means_se = []
        
        for n in num_samples_list:
            df_subset = entropy_df[(entropy_df["dataset"]==dataset)&(entropy_df["n"]==n)&(entropy_df["oracle"]>0)]
            ratios_hybrid = df_subset["cs-hybrid"]/df_subset["oracle"]
            ratios_plugin = df_subset["plugin"]/df_subset["oracle"]
            ratios_se = df_subset["se"]/df_subset["oracle"]

            means_hybrid.append(ratios_hybrid.mean())
            means_plugin.append(ratios_plugin.mean())
            means_se.append(ratios_se.mean())
        
        if aggregate_means_hybrid is None:
            aggregate_means_hybrid = np.array(means_hybrid)
        else:
            aggregate_means_hybrid += np.array(means_hybrid)

        if aggregate_means_plugin is None:
            aggregate_means_plugin = np.array(means_plugin)
        else:
            aggregate_means_plugin += np.array(means_plugin)

        if aggregate_means_se is None:
            aggregate_means_se = np.array(means_se)
        else:
            aggregate_means_se += np.array(means_se)

    aggregate_means_hybrid /= len(datasets)
    aggregate_means_plugin /= len(datasets)
    aggregate_means_se /= len(datasets)

    ax.plot(
        num_samples_list, aggregate_means_hybrid, 
        "o", markersize=12,
        label="$\\widehat{\\mathbb{H}}_{Hybrid}$ (Ours)",
        color=colors.indigo,
        linestyle=None,
        lw=4
    )

    ax.plot(
        num_samples_list, aggregate_means_plugin, 
        "x", markersize=12,
        label="$\\widehat{\\mathbb{H}}_{Plugin}$ (Canonical DSE)",
        color=colors.green,
        linestyle=":",
        lw=4
    )

    ax.plot(
        num_samples_list, aggregate_means_se, 
        "s", markersize=12,
        label="SE (White-Box)",
        color=colors.cyan,
        linestyle=":",
        lw=4
    )

    sns.despine(top=True, right=True, left=False, bottom=False, ax=ax)
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.tick_params(axis="y", which="both", right=False)
    ax.tick_params(axis="x", which="both", top=False) 

    ax.set_title(model_rename[model], fontsize=large_fontsize)
    ax.set_xlabel("$n$", fontsize=large_fontsize)
    if model_idx == 0:
        ax.set_ylabel("$\\langle\widehat{SE}/SE^*\\rangle$", fontsize=large_fontsize)
    ax.set_ylim(0.3, 1.2)
    ax.set_xscale("log")
    ax.set_xticks([5, 10, 25, 50, 100])
    ax.set_xticklabels([5, 10, 25, 50, 100])
    ax.set_yticks([0.4, 0.6, 0.8, 1.0, 1.2])
    ax.tick_params(axis='both', which='both', labelsize=small_fontsize)

handles, labels = ax.get_legend_handles_labels()
handles = [handles[1], handles[2], handles[3], handles[0]]
labels = [labels[1], labels[2], labels[3], labels[0]]
# interleave for 2 column setup
if square:
    handles = [handles[i] for i in range(1, len(handles), 2)]+[handles[i] for i in range(0, len(handles), 2)]
    labels = [labels[i] for i in range(1, len(labels), 2)]+[labels[i] for i in range(0, len(labels), 2)]
fig.legend(
    handles, labels, fontsize=medium_fontsize, loc='lower center', 
        bbox_to_anchor=(0.5, -0.35), ncol=4
)

plt.tight_layout()
plt.savefig('figures/entropy_ratios_plugin_summary_plus_se_draft.pdf')

#### Evaluation of entropy estimators

In [ ]:
def get_entropy_df(n, normalize=False):
    cols = ["Dataset", "Estimator"]
    for model in models:
        cols.append(f"{model_rename[model]}")
    df = pd.DataFrame(columns=cols)

    idx = 0
    num_methods = len(entropy_methods)
    for dataset in datasets:
        for m in entropy_methods:
            df.loc[idx, "Estimator"] = entropy_methods[m]
            for model in models:
                entropy_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/uncertainty.csv")
                entropy_df[m+"_SE"] = (entropy_df[m]-entropy_df["oracle"])**2
                          
                se_vals = entropy_df[(entropy_df["n"]==n)&(entropy_df["dataset"]==dataset)][m+"_SE"]

                if normalize:
                    # normalize by mean of target variable
                    mean_true = entropy_df[(entropy_df["n"]==n)&(entropy_df["dataset"]==dataset)]["oracle"].mean()
                    se_vals /= mean_true

                nmse = se_vals.mean()
                se_sem = se_vals.std()/np.sqrt(se_vals.count())
                se_interval = 1.96 * se_sem

                if idx % num_methods == num_methods - 1:
                    df.loc[idx, model_rename[model]] = f"\\textbf{{{nmse:.2f}}} \\textcolor{{gray}}{{{pm_symbol} {se_interval:.2f}}}"
                else:
                    df.loc[idx, model_rename[model]] = f"{nmse:.2f} \\textcolor{{gray}}{{{pm_symbol} {se_interval:.2f}}}"

                if idx % num_methods == 0:
                    dataset_label = datasets[dataset]
                    
                    df.loc[idx, "Dataset"] = f"\multirow{{{num_methods}}}{{*}}{{{dataset_label}}}"
                else:
                    df.loc[idx, "Dataset"] = ""
            idx += 1
    return df

#### MSE table with n=10

In [41]:
entropy_df = pd.read_csv(f"{current_dir}/data/{p}/{models[0]}/uncertainty.csv")
for m in entropy_methods:
    entropy_df[m+"_SE"] = (entropy_df[m]-entropy_df["oracle"])**2

In [42]:
label = "tab:entropy_10"
df_10 = get_entropy_df(n=10, normalize=False)
df_str = df_10.to_latex(
    index=False,
    float_format="{:.2f}".format,
)
df_str = f"""
\\begin{{table*}}[h!]
\\centering
""" + df_str + f"""
\\label{{{label}}}
\\end{{table*}}
"""
df_str = df_str.replace(
    "\\begin{tabular}{llllll}", "\\begin{tabular}{c|c|cccc}"
).replace(
    "\\toprule",""
).replace(
    "\\midrule","\\hline"
).replace(
    "\\bottomrule",""
).replace(
    "\\\\\n\\multirow","\\\\ \n\\hline\n\\multirow"
)
print(df_str)


\begin{table*}[h!]
\centering
\begin{tabular}{lllllll}

Dataset & Estimator & Gemma-2-9B & Gemma-3-12B & Llama-3.1-8B & Mistral-7B & Phi-3.5-3.8B \\
\hline
\multirow{3}{*}{HotpotQA} & $\widehat{\mathbb{H}}_{Plugin}$ & 0.46 \textcolor{gray}{± 0.02} & 0.09 \textcolor{gray}{± 0.01} & 0.68 \textcolor{gray}{± 0.03} & 0.59 \textcolor{gray}{± 0.03} & 0.61 \textcolor{gray}{± 0.03} \\
 & $\widehat{\mathbb{H}}_{CS-GT}$ & 0.39 \textcolor{gray}{± 0.02} & 0.08 \textcolor{gray}{± 0.01} & 0.56 \textcolor{gray}{± 0.03} & 0.46 \textcolor{gray}{± 0.03} & 0.47 \textcolor{gray}{± 0.03} \\
 & $\widehat{\mathbb{H}}_{Hybrid}$ & \textbf{0.30} \textcolor{gray}{± 0.02} & \textbf{0.06} \textcolor{gray}{± 0.01} & \textbf{0.45} \textcolor{gray}{± 0.02} & \textbf{0.39} \textcolor{gray}{± 0.02} & \textbf{0.39} \textcolor{gray}{± 0.02} \\ 
\hline
\multirow{3}{*}{SQuAD 2.0} & $\widehat{\mathbb{H}}_{Plugin}$ & 0.68 \textcolor{gray}{± 0.03} & 0.18 \textcolor{gray}{± 0.02} & 0.78 \textcolor{gray}{± 0.03} & 1.37 \textcol